# POI Data Classification - Enhanced Version

This notebook classifies Points of Interest (POI) into **seven main categories** using an optimized classification system:

- **Business**: Commercial establishments (shops, restaurants, cafes, retail buildings)
- **Tourism**: Tourism-related facilities (guest houses, hotels, museums, attractions)
- **Public Services**: Government and community services (police, schools, hospitals, social facilities)
- **Transportation**: Transportation infrastructure (roads, rails, traffic signals, parking)
- **Residential**: Housing and residential buildings (houses, apartments)
- **Infrastructure**: Utilities, landuse, and industrial facilities (farmland, forests, industrial buildings)
- **Other**: Uncategorized POIs

## Features
- ✅ Handles large datasets (1M+ records) efficiently
- ✅ Comprehensive coverage of Japanese POI data
- ✅ Optimized performance with set-based lookups
- ✅ Batch processing capabilities
- ✅ Progress tracking for large datasets

In [ ]:
# Import required libraries
import pandas as pd
import os
import sys

# Add project root to path to ensure imports work correctly
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Current working directory: {os.getcwd()}")

## 1. Load POI Data

Choose which dataset to process:
- **poi_data.csv**: Small sample dataset (17 records) - Good for testing
- **cleaned_JP_POI_part1.csv**: Large Japanese dataset Part 1 (508K records)
- **cleaned_JP_POI_part2.csv**: Large Japanese dataset Part 2 (508K records)
- **all**: Process all datasets combined (1M+ records)

In [ ]:
# Configuration: Select dataset to process
# Options: 'sample', 'part1', 'part2', 'all'
DATASET = 'sample'  # Change this to process different datasets

# Define available datasets
datasets = {
    'sample': 'poi_data.csv',
    'part1': 'cleaned_JP_POI_part1.csv',
    'part2': 'cleaned_JP_POI_part2.csv',
}

# Load data based on selection
if DATASET == 'all':
    print("Loading ALL datasets (this may take a while)...")
    dfs = []
    for name, filename in datasets.items():
        data_path = os.path.join(project_root, 'data', filename)
        if os.path.exists(data_path):
            print(f"  Loading {filename}...")
            df_temp = pd.read_csv(data_path)
            dfs.append(df_temp)
            print(f"    ✓ Loaded {len(df_temp):,} records")
    df = pd.concat(dfs, ignore_index=True)
    print(f"\n✓ Combined dataset: {len(df):,} total records")
else:
    filename = datasets.get(DATASET, 'poi_data.csv')
    data_path = os.path.join(project_root, 'data', filename)
    print(f"Loading dataset: {filename}")
    df = pd.read_csv(data_path)
    print(f"✓ Loaded {len(df):,} POI records")

print(f"\nData shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few records:")
df.head(10)

## 2. Import Enhanced Classification Function

We'll use the improved classification function from our module that supports 7 categories and comprehensive Japanese POI types.

In [ ]:
# Import the enhanced classifier from our module
from src.poi_classifier import classify_poi

# Test the function with various samples
print("Sample classifications:")
print("=" * 60)
test_cases = [
    ('amenity', 'restaurant', 'business'),
    ('tourism', 'guest_house', 'tourism'),
    ('amenity', 'police', 'public_services'),
    ('highway', 'traffic_signals', 'transportation'),
    ('building', 'house', 'residential'),
    ('landuse', 'farmland', 'infrastructure'),
    ('building', 'yes', 'infrastructure'),
    ('highway', 'residential', 'transportation'),
]

for poi_type, poi, expected in test_cases:
    result = classify_poi(poi_type, poi)
    status = "✓" if result == expected else "✗"
    print(f"{status} {poi_type:20} | {poi:20} → {result}")

## 3. Apply Classification to Dataset

In [ ]:
# Apply the classification function to all rows
print(f"Classifying {len(df):,} POI records...")
print("This may take a moment for large datasets...")

# For large datasets, show progress
import time
start_time = time.time()

df['category'] = df.apply(lambda row: classify_poi(row['POI_type'], row['POI']), axis=1)

elapsed = time.time() - start_time
print(f"\n✓ Classification completed in {elapsed:.2f} seconds")
print(f"  Processing rate: {len(df)/elapsed:,.0f} records/second")
print(f"\nDataFrame with classifications (first 20 rows):")
df.head(20)

## 4. Analyze Classification Results

In [ ]:
# Count POIs by category
category_counts = df['category'].value_counts().sort_values(ascending=False)
print("POI Count by Category:")
print("=" * 60)
for category, count in category_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  {category:20}: {count:>10,} ({percentage:>5.2f}%)")
print("=" * 60)
print(f"  {'Total':20}: {len(df):>10,}")
print()

In [ ]:
# Show percentage distribution with visualization
category_percentages = (df['category'].value_counts(normalize=True) * 100).round(2).sort_values(ascending=False)

print("Category Distribution (%):")
print("=" * 60)
for category, percentage in category_percentages.items():
    bar_length = int(percentage / 2)  # Scale for display
    bar = "█" * bar_length
    print(f"  {category:20}: {percentage:>6.2f}% {bar}")
print("=" * 60)

In [ ]:
# Group by category and show examples (limit for large datasets)
print("Examples by Category:")
print("=" * 80)

for category in sorted(df['category'].unique()):
    category_df = df[df['category'] == category][['POI_type', 'POI', 'latitude', 'longitude']]
    count = len(category_df)
    
    print(f"\n{category.upper()} ({count:,} records):")
    print("-" * 80)
    
    # Show first 10 examples for each category
    sample_size = min(10, count)
    print(category_df.head(sample_size).to_string(index=False))
    
    if count > sample_size:
        print(f"  ... and {count - sample_size:,} more records")
print()

## 5. Save Classified Data

In [ ]:
# Save the classified data to a new CSV file
output_filename = f"{DATASET}_classified.csv" if DATASET != 'sample' else 'poi_data_classified.csv'
output_path = os.path.join(project_root, 'data', output_filename)

print(f"Saving classified data to: {output_filename}")
df.to_csv(output_path, index=False)
print(f"✓ Successfully saved {len(df):,} classified records")
print(f"  File: {output_path}")

## 6. Summary Statistics

In [ ]:
# Create a comprehensive summary table
print("Summary Statistics by Category:")
print("=" * 100)

summary = df.groupby('category').agg({
    'POI': 'count',
    'latitude': ['min', 'max', 'mean'],
    'longitude': ['min', 'max', 'mean']
}).round(6)

summary.columns = ['Count', 'Min_Lat', 'Max_Lat', 'Avg_Lat', 'Min_Lon', 'Max_Lon', 'Avg_Lon']

# Add percentage column
summary['Percentage'] = (summary['Count'] / len(df) * 100).round(2)

# Reorder columns
summary = summary[['Count', 'Percentage', 'Min_Lat', 'Max_Lat', 'Avg_Lat', 'Min_Lon', 'Max_Lon', 'Avg_Lon']]

print(summary)
print("=" * 100)

# POI type distribution within each category
print("\nTop 5 POI Types per Category:")
print("=" * 100)
for category in sorted(df['category'].unique()):
    category_df = df[df['category'] == category]
    top_types = category_df.groupby(['POI_type', 'POI']).size().sort_values(ascending=False).head(5)
    
    print(f"\n{category.upper()}:")
    for (poi_type, poi), count in top_types.items():
        percentage = (count / len(category_df)) * 100
        print(f"  {poi_type:20} | {poi:25} : {count:>8,} ({percentage:>5.2f}%)")
print("=" * 100)

## 7. Data Quality Analysis

Let's analyze the data quality and identify any potential issues or interesting patterns.

In [ ]:
# Data quality checks
print("Data Quality Analysis:")
print("=" * 80)

# Check for missing values
print("\n1. Missing Values:")
print(df.isnull().sum())

# Check for unique POI types
print(f"\n2. Unique POI Types: {df['POI_type'].nunique()}")
print(f"   Unique POI Subtypes: {df['POI'].nunique()}")

# Geographic coverage
print(f"\n3. Geographic Coverage:")
print(f"   Latitude range:  {df['latitude'].min():.6f} to {df['latitude'].max():.6f}")
print(f"   Longitude range: {df['longitude'].min():.6f} to {df['longitude'].max():.6f}")

# Check distribution of 'other' category
other_count = len(df[df['category'] == 'other'])
if other_count > 0:
    print(f"\n4. Uncategorized POIs ('other' category): {other_count:,} ({other_count/len(df)*100:.2f}%)")
    print("   Top 10 POI types in 'other' category:")
    other_df = df[df['category'] == 'other']
    top_other = other_df.groupby(['POI_type', 'POI']).size().sort_values(ascending=False).head(10)
    for (poi_type, poi), count in top_other.items():
        print(f"     {poi_type:20} | {poi:25} : {count:>8,}")
else:
    print(f"\n4. ✓ All POIs successfully categorized (no 'other' category)")

print("\n5. Classification Coverage:")
categorized = len(df[df['category'] != 'other'])
coverage = (categorized / len(df)) * 100
print(f"   Successfully categorized: {categorized:,} / {len(df):,} ({coverage:.2f}%)")
print("=" * 80)

print("\n✓ Analysis complete! The classifier is working with the new data.")
print(f"  You can change DATASET to 'part1', 'part2', or 'all' to process larger datasets.")